# MVP de Machine Learning e Analytics

## Clusterizacao de fundos de investimento por perfil de risco e performance

Este notebook segue a estrutura do template do MVP e propõe uma solucao simples, util e reproduzivel para a rotina de uma gestora de fundos.

A ideia e usar dados publicos da CVM para agrupar fundos com comportamento semelhante em risco, retorno, patrimonio e fluxo. A tarefa escolhida e **clusterizacao**, pois nao existe variavel-alvo supervisionada.

## 1. Apresentacao do problema

O objetivo deste MVP e segmentar fundos de investimento em grupos interpretaveis, apoiando comparacao de pares, leitura de risco e analise de comportamento historico.

A proposta usa apenas dados publicos e pode ser executada no Colab do inicio ao fim sem upload manual ou credenciais.

In [ ]:
import io
import os
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.cluster import AgglomerativeClustering, KMeans
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.metrics import calinski_harabasz_score, davies_bouldin_score, silhouette_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 120)
np.random.seed(42)

BASE_DIR = Path('cvm_mvp_data')
BASE_DIR.mkdir(exist_ok=True)
RAW_DIR = BASE_DIR / 'raw'
RAW_DIR.mkdir(exist_ok=True)

print('Ambiente pronto.')

## 2. Apresentacao dos dados

A base principal e a serie diaria mensal da CVM (`inf_diario_fi_YYYYMM.zip`), combinada com o registro de classe (`registro_fundo_classe.zip`).

Arquivos usados neste recorte:
- `https://dados.cvm.gov.br/dados/FI/DOC/INF_DIARIO/DADOS/`
- `https://dados.cvm.gov.br/dados/FI/CAD/DADOS/registro_fundo_classe.zip`

A janela sugerida vai de 2024-01 ate 2026-06.

In [ ]:
import requests

def month_range(start='2024-01', end='2026-06'):
    return pd.period_range(start=start, end=end, freq='M').strftime('%Y%m').tolist()

def download_file(url, out_path):
    if not out_path.exists():
        response = requests.get(url, timeout=60)
        response.raise_for_status()
        out_path.write_bytes(response.content)
    return out_path

def load_daily_month(zip_path):
    with zipfile.ZipFile(zip_path) as zf:
        csv_name = [name for name in zf.namelist() if name.endswith('.csv')][0]
        with zf.open(csv_name) as handle:
            df = pd.read_csv(
                handle,
                sep=';',
                dtype={
                    'TP_FUNDO_CLASSE': 'string',
                    'CNPJ_FUNDO_CLASSE': 'string',
                    'ID_SUBCLASSE': 'string'
                },
                usecols=['TP_FUNDO_CLASSE', 'CNPJ_FUNDO_CLASSE', 'ID_SUBCLASSE', 'DT_COMPTC', 'VL_TOTAL', 'VL_QUOTA', 'VL_PATRIM_LIQ', 'CAPTC_DIA', 'RESG_DIA', 'NR_COTST'],
                low_memory=False
            )
    return df

daily_frames = []
for ym in month_range():
    url = f'https://dados.cvm.gov.br/dados/FI/DOC/INF_DIARIO/DADOS/inf_diario_fi_{ym}.zip'
    zip_path = download_file(url, RAW_DIR / f'inf_diario_fi_{ym}.zip')
    daily_frames.append(load_daily_month(zip_path))

daily = pd.concat(daily_frames, ignore_index=True)
daily['DT_COMPTC'] = pd.to_datetime(daily['DT_COMPTC'], errors='coerce')
for col in ['VL_TOTAL', 'VL_QUOTA', 'VL_PATRIM_LIQ', 'CAPTC_DIA', 'RESG_DIA', 'NR_COTST']:
    daily[col] = pd.to_numeric(daily[col], errors='coerce')

print(daily.shape)
daily.head()

In [ ]:
registry_zip = download_file('https://dados.cvm.gov.br/dados/FI/CAD/DADOS/registro_fundo_classe.zip', RAW_DIR / 'registro_fundo_classe.zip')
with zipfile.ZipFile(registry_zip) as zf:
    csv_name = [name for name in zf.namelist() if name.endswith('registro_classe.csv')][0]
    with zf.open(csv_name) as handle:
        registry = pd.read_csv(handle, sep=';', dtype={'CNPJ_Classe': 'string'}, low_memory=False)

registry['Data_Registro'] = pd.to_datetime(registry['Data_Registro'], errors='coerce')
registry['Data_Constituicao'] = pd.to_datetime(registry['Data_Constituicao'], errors='coerce')
registry['Data_Inicio'] = pd.to_datetime(registry['Data_Inicio'], errors='coerce')
registry['Data_Patrimonio_Liquido'] = pd.to_datetime(registry['Data_Patrimonio_Liquido'], errors='coerce')

df = daily.merge(registry, left_on='CNPJ_FUNDO_CLASSE', right_on='CNPJ_Classe', how='left')
print(df.shape)
df[['CNPJ_FUNDO_CLASSE', 'DT_COMPTC', 'VL_QUOTA', 'VL_PATRIM_LIQ', 'Tipo_Classe', 'Denominacao_Social']].head()

## 3. Analise exploratoria inicial

Nesta etapa, vamos verificar cobertura temporal, valores ausentes e distribuicoes basicas para entender a qualidade da base antes da modelagem.

In [ ]:
summary = pd.DataFrame({
    'n_linhas': [len(df)],
    'n_fundos': [df['CNPJ_FUNDO_CLASSE'].nunique()],
    'data_inicial': [df['DT_COMPTC'].min()],
    'data_final': [df['DT_COMPTC'].max()]
})
display(summary)
display(df.isna().mean().sort_values(ascending=False).head(12).to_frame('taxa_ausentes'))
display(df[['VL_QUOTA', 'VL_PATRIM_LIQ', 'CAPTC_DIA', 'RESG_DIA', 'NR_COTST']].describe().T)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(df['VL_PATRIM_LIQ'].dropna(), bins=50, ax=axes[0], kde=True)
axes[0].set_title('Distribuicao do patrimonio liquido')
sns.histplot(df['VL_QUOTA'].dropna(), bins=50, ax=axes[1], kde=True)
axes[1].set_title('Distribuicao da cota')
plt.tight_layout()

## 4. Preparacao dos dados

Como o problema e nao supervisionado, a preparacao e orientada a criar uma matriz de features consistente por fundo.

As principais variaveis derivadas sao retorno diario, volatilidade, drawdown, fluxo liquido, variacao de patrimonio e numero medio de cotistas.

In [ ]:
df = df.sort_values(['CNPJ_FUNDO_CLASSE', 'DT_COMPTC']).copy()
df['retorno_diario'] = df.groupby('CNPJ_FUNDO_CLASSE')['VL_QUOTA'].pct_change()
df['variacao_patrimonio'] = df.groupby('CNPJ_FUNDO_CLASSE')['VL_PATRIM_LIQ'].pct_change()
df['fluxo_liquido'] = df['CAPTC_DIA'] - df['RESG_DIA']
df['drawdown'] = df['VL_QUOTA'] / df.groupby('CNPJ_FUNDO_CLASSE')['VL_QUOTA'].cummax() - 1

min_obs = 60
df_recent = (
    df.groupby('CNPJ_FUNDO_CLASSE', group_keys=False)
      .filter(lambda x: x['DT_COMPTC'].notna().sum() >= min_obs)
      .sort_values(['CNPJ_FUNDO_CLASSE', 'DT_COMPTC'])
)

def fund_features(group):
    group = group.dropna(subset=['VL_QUOTA'])
    retorno = group['retorno_diario'].dropna()
    variacao_patrimonio = group['variacao_patrimonio'].dropna()
    return pd.Series({
        'qtd_obs': len(group),
        'retorno_medio': retorno.mean(),
        'retorno_total': (group['VL_QUOTA'].iloc[-1] / group['VL_QUOTA'].iloc[0]) - 1 if len(group) > 1 else np.nan,
        'volatilidade_diaria': retorno.std(),
        'volatilidade_anualizada': retorno.std() * np.sqrt(252),
        'max_drawdown': group['drawdown'].min(),
        'drawdown_medio': group['drawdown'].mean(),
        'patrimonio_medio': group['VL_PATRIM_LIQ'].mean(),
        'patrimonio_final': group['VL_PATRIM_LIQ'].iloc[-1],
        'crescimento_patrimonio': (group['VL_PATRIM_LIQ'].iloc[-1] / group['VL_PATRIM_LIQ'].iloc[0]) - 1 if len(group) > 1 else np.nan,
        'fluxo_liquido_medio': group['fluxo_liquido'].mean(),
        'cotistas_medios': group['NR_COTST'].mean(),
        'variacao_patrimonio_media': variacao_patrimonio.mean(),
    })

features = df_recent.groupby('CNPJ_FUNDO_CLASSE').apply(fund_features).reset_index()
meta = df_recent.groupby('CNPJ_FUNDO_CLASSE').agg(
    Denominacao_Social=('Denominacao_Social', 'first'),
    Tipo_Classe=('Tipo_Classe', 'first'),
    Classificacao=('Classificacao', 'first'),
    Publico_Alvo=('Publico_Alvo', 'first'),
    Forma_Condominio=('Forma_Condominio', 'first')
).reset_index()

model_df = features.merge(meta, on='CNPJ_FUNDO_CLASSE', how='left')
model_df = model_df.replace([np.inf, -np.inf], np.nan)
print(model_df.shape)
display(model_df.head())

## 5. Baseline e modelagem

A baseline e uma segmentacao simples por quartis de volatilidade. Depois vamos comparar K-Means e Agglomerative Clustering.

In [ ]:
numeric_cols = [
    'retorno_medio', 'retorno_total', 'volatilidade_diaria', 'volatilidade_anualizada',
    'max_drawdown', 'drawdown_medio', 'patrimonio_medio', 'patrimonio_final',
    'crescimento_patrimonio', 'fluxo_liquido_medio', 'cotistas_medios', 'variacao_patrimonio_media'
]

data_model = model_df[['CNPJ_FUNDO_CLASSE'] + numeric_cols].copy()
imputer = SimpleImputer(strategy='median')
scaler = StandardScaler()
X = scaler.fit_transform(imputer.fit_transform(data_model[numeric_cols]))

baseline = data_model.copy()
baseline['baseline_risco'] = pd.qcut(baseline['volatilidade_anualizada'].rank(method='first'), q=4, labels=['Baixo', 'Moderado', 'Alto', 'Muito alto'])
display(baseline[['CNPJ_FUNDO_CLASSE', 'baseline_risco']].head())

k_values = range(2, 9)
silhouette_values = []
inertia_values = []
for k in k_values:
    km = KMeans(n_clusters=k, random_state=42, n_init='auto')
    labels = km.fit_predict(X)
    silhouette_values.append(silhouette_score(X, labels))
    inertia_values.append(km.inertia_)

best_k = list(k_values)[int(np.argmax(silhouette_values))]
print('Melhor k sugerido pelo silhouette:', best_k)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(list(k_values), inertia_values, marker='o')
axes[0].set_title('Elbow method')
axes[0].set_xlabel('k')
axes[0].set_ylabel('Inercia')
axes[1].plot(list(k_values), silhouette_values, marker='o', color='darkgreen')
axes[1].set_title('Silhouette por k')
axes[1].set_xlabel('k')
axes[1].set_ylabel('Silhouette')
plt.tight_layout()

kmeans = KMeans(n_clusters=best_k, random_state=42, n_init='auto')
kmeans_labels = kmeans.fit_predict(X)
agg = AgglomerativeClustering(n_clusters=best_k)
agg_labels = agg.fit_predict(X)

results = pd.DataFrame({
    'modelo': ['KMeans', 'Agglomerative'],
    'silhouette': [silhouette_score(X, kmeans_labels), silhouette_score(X, agg_labels)],
    'davies_bouldin': [davies_bouldin_score(X, kmeans_labels), davies_bouldin_score(X, agg_labels)],
    'calinski_harabasz': [calinski_harabasz_score(X, kmeans_labels), calinski_harabasz_score(X, agg_labels)]
})
display(results.sort_values('silhouette', ascending=False))

In [ ]:
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X)
viz = model_df[['CNPJ_FUNDO_CLASSE', 'Denominacao_Social', 'Tipo_Classe'] + numeric_cols].copy()
viz['cluster_kmeans'] = kmeans_labels
viz['cluster_agg'] = agg_labels
viz['pca_1'] = X_pca[:, 0]
viz['pca_2'] = X_pca[:, 1]

plt.figure(figsize=(10, 7))
sns.scatterplot(data=viz, x='pca_1', y='pca_2', hue='cluster_kmeans', palette='tab10', s=60, alpha=0.85)
plt.title('Clusters do K-Means em 2D via PCA')
plt.legend(title='Cluster')
plt.tight_layout()

cluster_profile = model_df.copy()
cluster_profile['cluster_kmeans'] = kmeans_labels
profile = cluster_profile.groupby('cluster_kmeans')[numeric_cols].mean().round(4)
display(profile)

## 6. Avaliacao dos resultados

A avaliacao principal usa metricas internas de clusterizacao, ja que nao existe alvo supervisionado.

As metricas principais sao silhouette, Davies-Bouldin e Calinski-Harabasz. Alem disso, a interpretacao economica de cada grupo e essencial para validar se os clusters fazem sentido na pratica.

In [ ]:
model_df['cluster_kmeans'] = kmeans_labels
model_df['cluster_agg'] = agg_labels

cluster_summary = model_df.groupby('cluster_kmeans')[numeric_cols].agg(['mean', 'median', 'count'])
display(cluster_summary)

counts = model_df['cluster_kmeans'].value_counts().sort_index()
plt.figure(figsize=(8, 4))
counts.plot(kind='bar', color='steelblue')
plt.title('Quantidade de fundos por cluster')
plt.xlabel('Cluster')
plt.ylabel('Quantidade')
plt.tight_layout()

## 7. Conclusao

Este MVP propôs a clusterizacao de fundos de investimento a partir de dados publicos da CVM. A soluçao e util para leitura de risco, comparaçao de pares e segmentaçao de uma carteira de fundos.

O fluxo cobre coleta, limpeza, engenharia de atributos, baseline, K-Means, Agglomerative Clustering, tunning simples de k e avaliacao por metricas internas.

Como proximos passos, vale testar outras janelas temporais, incluir benchmark macro e avaliar estabilidade dos clusters ao longo do tempo.